### Univariate Analysis

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from typing import Dict, Any, cast
from matplotlib.patches import Rectangle

def numeric_timeseries_univariate_analysis(s: pl.Series) -> Dict[str, Any]:
    """
    Brutally optimized univariate analysis for a numeric Time Series.
    Performance Architecture:
     - Parallel Execution: Computes full numeric stats (skew, kurtosis, percentiles) 
       in a single parallel pass over the entire dataset.
     - Strided Downsampling: Uses deterministic skipping (gather) rather than random 
       sampling to preserve the true chronological shape without freezing Seaborn.
    """
    warnings.filterwarnings("ignore")
    name = s.name if s.name else "value"
    total_rows = len(s)
    if total_rows == 0:
        raise ValueError("Series is empty.")
        
    # 1. Parallel Statistical Computation (Exact Numeric Stats)
    df = s.to_frame()
    stats = df.select([
        pl.len().alias("total_rows"),
        pl.col(name).null_count().alias("null_count"),
        pl.col(name).n_unique().alias("unique"),
        pl.col(name).mean().alias("mean"),
        pl.col(name).median().alias("median"),
        pl.col(name).min().alias("min"),
        pl.col(name).max().alias("max"),
        pl.col(name).var().alias("var"),
        pl.col(name).std().alias("std"),
        pl.col(name).skew().alias("skew"),
        pl.col(name).kurtosis().alias("kurtosis"),
        pl.col(name).quantile(0.25).alias("p25"),
        pl.col(name).quantile(0.50).alias("p50"),
        pl.col(name).quantile(0.75).alias("p75"),
        pl.col(name).quantile(0.99).alias("p99"),
    ])
    
    stats_dict: Dict[str, Any] = stats.to_dicts()[0]
    null_count = int(stats_dict["null_count"])
    stats_dict["null_pct"] = (null_count / total_rows) * 100
    
    # Mode and Mode Percentage via Polars parallel hash aggregation
    vc = s.value_counts(sort=True)
    if vc.height > 0:
        row_data = vc.row(0)
        mode_val = row_data[0]
        mode_count = int(row_data[1])
        stats_dict["mode"] = mode_val
        stats_dict["mode_pct"] = (mode_count / total_rows) * 100
    else:
        stats_dict["mode"] = None
        stats_dict["mode_pct"] = 0.0
        
    # 2. Smart Strided Downsampling (Chronology Preservation)
    # We append a row index to act as our Time Step (X-axis)
    df_indexed = df.with_row_index(name="time_step")
    df_clean = df_indexed.drop_nulls(subset=[name])
    
    if s.dtype in [pl.Float32, pl.Float64]:
        df_clean = df_clean.filter(pl.col(name).is_not_nan())
        
    clean_len = df_clean.height
    
    # Max out at ~250k points to prevent rendering freezes, keeping chronological order
    step_size = max(1, clean_len // 250_000)
    if step_size > 1 and clean_len > 0:
        indices = pl.int_range(0, clean_len, step=step_size, dtype=pl.UInt32)
        df_plot = df_clean.gather(indices)
    else:
        df_plot = df_clean
        
    time_steps = df_plot.get_column("time_step").to_numpy()
    vals_array = df_plot.get_column(name).to_numpy()
        
    # 3. Modern Business Presentation Plotting
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.5, 1]})
    fig.patch.set_facecolor('#F8F9FA')
    for ax in axes:
        ax.set_facecolor('#F8F9FA')
        
    def safe_fmt(val: Any, decimals: int = 4) -> str:
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return "N/A"
        return f"{val:.{decimals}f}"
        
    if len(vals_array) > 0:
        # Left Plot: Chronological Trend (Index as Time)
        sns.lineplot(x=time_steps, y=vals_array, ax=axes[0], 
                     color='#E63946', linewidth=1.5)
        axes[0].fill_between(time_steps, vals_array, color='#E63946', alpha=0.1)
        
        # Overlay Mean & Median on the time series
        mean_val = stats_dict["mean"]
        median_val = stats_dict["median"]
        axes[0].axhline(mean_val, color='#2A9D8F', linestyle='--', linewidth=2, label=f'Mean: {safe_fmt(mean_val)}')
        axes[0].axhline(median_val, color='#F4A261', linestyle='-.', linewidth=2, label=f'Median: {safe_fmt(median_val)}')
        
        axes[0].set_title(f"Sequential Trend of '{name}'", fontsize=16, fontweight='bold', color='#1D3557', pad=15)
        axes[0].set_xlabel("Time Step (Row Index)", fontsize=13, fontweight='bold', color='#457B9D')
        axes[0].set_ylabel(name, fontsize=13, fontweight='bold', color='#457B9D')
        axes[0].legend(fontsize=11, frameon=True, shadow=True)
        
        # Right Plot: Value Distribution Shape
        sns.histplot(vals_array, kde=True, ax=axes[1], color='#0077B6', 
                     line_kws={'color': '#1D3557', 'linewidth': 2}, bins=50, alpha=0.7)
        axes[1].set_title(f"Distribution of '{name}'", fontsize=16, fontweight='bold', color='#1D3557', pad=15)
        axes[1].set_xlabel(name, fontsize=13, fontweight='bold', color='#457B9D')
        axes[1].set_ylabel("Frequency", fontsize=13, fontweight='bold', color='#457B9D')
    else:
        axes[0].text(0.5, 0.5, 'No valid data to plot', ha='center', va='center', fontsize=14)
        axes[1].text(0.5, 0.5, 'No valid data to plot', ha='center', va='center', fontsize=14)
        
    # Clean up spines for business look
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#CCCCCC')
        ax.spines['bottom'].set_color('#CCCCCC')
        ax.tick_params(colors='#555555', labelsize=10)
        
    plt.tight_layout()
    plt.show()
    
    # 4. Beautiful Console Output
    print(f"\n{'='*55}")
    print(f"  UNIVARIATE ANALYSIS (TIME SERIES): {name.upper()}")
    print(f"{'='*55}")
    print(f"  Total Rows:        {total_rows:>15,}")
    print(f"  Null Count:        {null_count:>15,} ({stats_dict['null_pct']:.2f}%)")
    print(f"  Unique Values:     {stats_dict['unique']:>15,}")
    print(f"  Mean:              {safe_fmt(stats_dict['mean']):>15}")
    print(f"  Median:            {safe_fmt(stats_dict['median']):>15}")
    print(f"  Mode:              {safe_fmt(stats_dict['mode'])} ({stats_dict['mode_pct']:.2f}%)")
    print(f"  Min:               {safe_fmt(stats_dict['min']):>15}")
    print(f"  Max:               {safe_fmt(stats_dict['max']):>15}")
    print(f"  Variance:          {safe_fmt(stats_dict['var']):>15}")
    print(f"  Std Dev:           {safe_fmt(stats_dict['std']):>15}")
    print(f"  Skewness:          {safe_fmt(stats_dict['skew']):>15}")
    print(f"  Kurtosis:          {safe_fmt(stats_dict['kurtosis']):>15}")
    print(f"  25th Percentile:   {safe_fmt(stats_dict['p25']):>15}")
    print(f"  50th Percentile:   {safe_fmt(stats_dict['p50']):>15}")
    print(f"  75th Percentile:   {safe_fmt(stats_dict['p75']):>15}")
    print(f"  99th Percentile:   {safe_fmt(stats_dict['p99']):>15}")
    print(f"{'='*55}\n")
    
    return stats_dict

### Load Datasets

In [ ]:
north_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/north_raw.parquet")
south_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/south_raw.parquet")

### Get Column Names

In [ ]:
print(north_df.columns)

### Seasons

In [ ]:
print(categorical_univariate_analysis(s=north_df["season_label"]))
print(categorical_univariate_analysis(s=south_df["season_label"]))

### Daily Max Temperature

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["daily_max_temp_celsius"]))
print(numeric_timeseries_univariate_analysis(s=south_df["daily_max_temp_celsius"]))

### Temperature Anomaly

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["temp_anomaly_celsius"]))
print(numeric_timeseries_univariate_analysis(s=south_df["temp_anomaly_celsius"]))

### Cumulative Heat Index

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cumulative_heat_index"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cumulative_heat_index"]))

### Daily Rainfall

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["daily_rainfall_mm"]))
print(numeric_timeseries_univariate_analysis(s=south_df["daily_rainfall_mm"]))

### Consecutive Dry Days

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["consecutive_dry_days"]))
print(numeric_timeseries_univariate_analysis(s=south_df["consecutive_dry_days"]))

### Rolling Rainfall MM for 7 Days

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["rolling_7d_rainfall_mm"]))
print(numeric_timeseries_univariate_analysis(s=south_df["rolling_7d_rainfall_mm"]))

### Cumulative Storm Rainfall

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cumulative_storm_rainfall_mm"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cumulative_storm_rainfall_mm"]))

### Antecedent Moisture Condition

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["antecedent_moisture_condition"]))
print(numeric_timeseries_univariate_analysis(s=south_df["antecedent_moisture_condition"]))

### Daily Runoff Volume

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["daily_runoff_volume_m3"]))
print(numeric_timeseries_univariate_analysis(s=south_df["daily_runoff_volume_m3"]))

### Total Suspended Solids

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["total_suspended_solids_mg_L"]))
print(numeric_timeseries_univariate_analysis(s=south_df["total_suspended_solids_mg_L"]))

### Nutrient Load Index

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["nutrient_load_index"]))
print(numeric_timeseries_univariate_analysis(s=south_df["nutrient_load_index"]))